# RAG-поиск по научным публикациям о визуальном поиске товаров

Лабораторная работа № 3, вариант 2. Корпус охватывает работы о поиске
товаров по фотографии и сопоставлении изображений одного товара. Для
каталога IKEA такая постановка соответствует поиску мебели и предметов
интерьера по снимку пользователя. Здесь поисковая система возвращает
научные статьи о методах, а локальная языковая модель отвечает по их
заголовкам и аннотациям.

Откройте ноутбук из корня проекта, выберите Python из `.venv` и запустите
все ячейки. Порядок установки и устройство сохранённых данных описаны в
`README.md`.

## Imports

Стандартная библиотека обслуживает метаданные arXiv, пути и кэш.
Requests выполняет HTTP-запросы, NumPy и pandas обрабатывают матрицы и
таблицы, `tqdm` показывает ход расчётов. LangChain используется для
одной из стратегий чанкинга и формата ответа адаптера RAGAS. Остальные
пакеты нужны для векторизации, поиска, коррекции опечаток и локальной
генерации.

In [1]:
import gc
import hashlib
import json
import os
import random
import re
import sqlite3
import time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from huggingface_hub import snapshot_download
from IPython.display import Markdown, display
from langchain_core.outputs import Generation, LLMResult
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas import SingleTurnSample
from ragas.embeddings import BaseRagasEmbeddings
from ragas.exceptions import RagasException
from ragas.llms import BaseRagasLLM
from ragas.metrics import (
    Faithfulness,
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ResponseRelevancy,
)
from ragas.run_config import RunConfig
from rapidfuzz import process
from rapidfuzz.distance import DamerauLevenshtein
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

C:\Users\Vlad\PycharmProjects\RAG_System_IKEA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

| Параметр                                                   | Назначение                                                                                                                                                         |
|------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `PATHS.data`, `PATHS.models`                               | Каталоги данных и весов относительно корня проекта.                                                                                                                |
| `CORPUS.size`                                              | Число уникальных статей после фильтрации.                                                                                                                          |
| `CORPUS.page_size`                                         | Верхняя граница числа записей на страницу arXiv.                                                                                                                   |
| `CORPUS.stop_date`                                         | Верхняя граница даты первой подачи в формате arXiv, UTC.                                                                                                           |
| `CORPUS.interval`, `CORPUS.timeout`, `CORPUS.attempts`     | Минимальный интервал между запросами и таймаут в секундах, число попыток.                                                                                          |
| `ARXIV_QUERIES`                                            | Запросы в порядке приоритета от поисковых задач в торговле до смежных методов.                                                                                     |
| `CHUNKS.size`, `CHUNKS.overlap`                            | Целевой размер и перекрытие в словах.                                                                                                                              |
| `SPELLING.min_length`, `SPELLING.min_frequency`            | Минимальные длина и частота слова, предлагаемого для исправления.                                                                                                  |
| `MODELS.*.id`, `MODELS.*.revision`                         | Репозиторий модели и точная ревизия весов на Hugging Face.                                                                                                         |
| `MODELS.embedding_batch`                                   | Число текстов в пакете векторизации.                                                                                                                               |
| `MODELS.answer_tokens`, `MODELS.judge_tokens`              | Верхние границы длины ответа в новых токенах.                                                                                                                      |
| `MODELS.cpu_threads`                                       | Число потоков PyTorch для работы на CPU.                                                                                                                           |
| `EVALUATION.title_queries`, `EVALUATION.typos`             | Размер набора заголовков и число искажённых запросов.                                                                                                              |
| `EVALUATION.retrieval_depths`                              | Глубины для оценки поиска.                                                                                                                                         |
| `EVALUATION.generation_depths`, `EVALUATION.default_depth` | Глубины абляции и глубина основной демонстрации.                                                                                                                   |
| `EVALUATION.answerable`                                    | Число вопросов с ответом для RAGAS. К ним добавляются четыре проверочных вопроса без ответа.                                                                       |
| `EVALUATION.relevancy_strictness`                          | Число обратных вопросов для Answer Relevancy. Используется один вопрос на ответ, так как повтор одного промпта при жадном декодировании дал бы ту же формулировку. |
| `SEED`                                                     | Общий seed для выборок и вычислений.                                                                                                                               |
| `ABSTENTION`                                               | Точная фраза отказа по ТЗ.                                                                                                                                         |

Корпус, модель эмбеддингов, генератор и судья задаются отдельно. Выбор
устройства делается по доступности CUDA. Этот блок задаёт параметры и
направляет кэш Hugging Face в `models/`.

In [2]:
PATHS = {"data": Path("data"), "models": Path("models")}
CORPUS = {
    "size": 300,
    "page_size": 75,
    "stop_date": "202611162359",
    "interval": 3.2,
    "timeout": 35,
    "attempts": 3,
}
ARXIV_QUERIES = (
    '(abs:"furniture" OR abs:"home decor" OR abs:"interior design") '
    'AND (abs:"image retrieval" OR abs:"visual search" OR abs:"recognition" '
    'OR abs:"shape retrieval")',
    '(abs:"product retrieval" OR abs:"product matching" OR '
    'abs:"visual shopping" OR abs:"fashion retrieval" OR '
    'abs:"product recognition" OR abs:"product image retrieval" OR '
    'abs:"visual product search" OR abs:"product classification" OR '
    'abs:"fashion image retrieval")',
    '(abs:"e-commerce" OR abs:"online shopping" OR abs:"retail" OR '
    'abs:"product images" OR abs:"fashion") AND (abs:"image retrieval" '
    'OR abs:"image search" OR abs:"visual search" OR '
    'abs:"fine-grained recognition" OR abs:"object recognition" OR '
    'abs:"image similarity" OR abs:"visual similarity" OR '
    'abs:"image classification" OR abs:"product identification" OR '
    'abs:"multimodal retrieval")',
    '(abs:"instance-level image retrieval" OR '
    'abs:"fine-grained image retrieval" OR '
    'abs:"fine-grained visual recognition" OR '
    'abs:"sketch-based image retrieval")',
)
CHUNKS = {"size": 130, "overlap": 25}
SPELLING = {"min_length": 5, "min_frequency": 3}
MODELS = {
    "embedding": {
        "id": "BAAI/bge-small-en-v1.5",
        "revision": "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",
    },
    "answer": {
        "id": "Qwen/Qwen2.5-3B-Instruct",
        "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
    },
    "judge": {
        "id": "Qwen/Qwen2.5-3B-Instruct",
        "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
    },
    "embedding_batch": 32,
    "answer_tokens": 144,
    "judge_tokens": 448,
    "cpu_threads": 4,
}
EVALUATION = {
    "title_queries": 100,
    "typos": 40,
    "retrieval_depths": (1, 3, 5, 10),
    "generation_depths": (1, 3, 5),
    "default_depth": 3,
    "answerable": 16,
    "relevancy_strictness": 1,
}
SEED = 42
ABSTENTION = "Информация в найденных статьях отсутствует"

os.environ["HF_HOME"] = str((PATHS["models"] / ".cache").resolve())
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

## Seed

`random.seed` задаёт порядок выборки статей и генерации опечаток, NumPy
получает тот же seed. `RNG` выделяет независимый генератор для каждой
выборки, поэтому частично заполненный кэш не меняет её порядок.
PyTorch использует общий seed на CPU и на всех видимых
CUDA-устройствах. Детерминированные операции запрашиваются с предупреждением
для операций без такой реализации. Qwen использует жадное декодирование
или детерминированный поиск по лучам для метрики с несколькими ответами.
Воспроизводимость численных результатов проверяется на одной машине.

In [3]:
random.seed(SEED)
RNG = {
    "titles": random.Random(SEED),
    "typos": random.Random(SEED + 1),
    "questions": random.Random(SEED + 2),
}
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.benchmark = False
torch.set_num_threads(MODELS["cpu_threads"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
run_started = time.perf_counter()
print("PyTorch:", torch.__version__, "| Устройство:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name())

PyTorch: 2.8.0+cu128 | Устройство: cuda
NVIDIA GeForce RTX 4060 Laptop GPU


## Хранение промежуточных результатов

`fingerprint` связывает сохранённые результаты с входами. `artifact_dir`
создаёт отдельный каталог для версии этапа. `write_rows` обновляет JSONL
заменой временного файла, а `cached_rows` читает его или вызывает
переданную функцию расчёта. `local_model` находит веса нужной ревизии
в `models/` либо скачивает их в этот же каталог. Файл `.revision` даёт
возможность загрузить модель без обращения к Hugging Face.

In [4]:
def fingerprint(*values):
    payload = json.dumps(values, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def artifact_dir(stage, key):
    folder = PATHS["data"] / stage / key
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def write_rows(path, rows):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False, allow_nan=False))
            stream.write("\n")
    temporary.replace(path)


def cached_rows(path, build):
    if path.is_file():
        print("Из кэша:", path)
        with path.open(encoding="utf-8") as stream:
            return [json.loads(line) for line in stream if line.strip()]

    rows = build()
    write_rows(path, rows)
    print("Сохранено:", path)
    return rows


def local_model(spec):
    path = PATHS["models"] / spec["id"].split("/")[-1]
    marker = path / ".revision"
    hub_metadata = path / ".cache/huggingface/download/config.json.metadata"
    markers = (marker, hub_metadata)
    have_revision = any(
        file.is_file()
        and file.read_text(encoding="utf-8").startswith(spec["revision"])
        for file in markers
    )
    weight_index = path / "model.safetensors.index.json"
    required = {"config.json", "tokenizer.json", "model.safetensors"}
    if weight_index.is_file():
        weights = json.loads(weight_index.read_text(encoding="utf-8"))
        required.remove("model.safetensors")
        required.update(weights["weight_map"].values())

    if not have_revision or not all((path / f).is_file() for f in required):
        snapshot_download(
            repo_id=spec["id"],
            revision=spec["revision"],
            local_dir=path,
            cache_dir=PATHS["models"] / ".cache/hub",
            allow_patterns=["*.json", "*.txt", "*.safetensors"],
            token=False,
        )
        marker.write_text(spec["revision"], encoding="utf-8")
    return path

## 1. Сбор корпуса из arXiv

`parse_arxiv` переводит Atom-выдачу в единый формат. Идентификатор arXiv
без суффикса версии используется для дедупликации. `request_page` делает
один запрос с ограничением частоты, таймаутом и повтором временных ошибок.
`collect_corpus` запрашивает страницы последовательно, отдаёт приоритет
публикациям о товарах и мебели и прекращает сбор после набора 300 статей.
Категории `cs.CV`, `cs.MM` и `cs.IR` ограничивают выборку по тематике.
Сохраняются только записи со всеми обязательными полями. Дата отсечения
и порядок сортировки делают выборку объяснимой. Воспроизводимый снимок
корпуса хранится в JSONL. [Ограничения arXiv API](https://info.arxiv.org/help/api/tou.html).

`fits_scope` проверяет задачу по заголовку и аннотации. Для основных
запросов должны одновременно встречаться признаки работы с изображениями,
поиска или распознавания и предметной области. Для смежных методов
допускается работа по поиску изображений или точному визуальному распознаванию,
когда задача прямо указана в названии.

In [5]:
ARXIV_ENDPOINT = "https://export.arxiv.org/api/query"
ATOM = {"a": "http://www.w3.org/2005/Atom"}
VISION = re.compile(
    r"\b(image\w*|visual\w*|vision|photo\w*|pictures?|sketch\w*|3d)\b", re.I
)
TASK = re.compile(
    r"\b(retriev\w*|search\w*|match\w*|recogni\w*|classifi\w*|similar\w*)\b",
    re.I,
)
RETAIL = re.compile(
    r"\b(e-commerce|ecommerce|online shopping|online store|retail\w*|"
    r"product (?:images?|retrieval|matching|recognition|"
    r"classification|search|catalog\w*)|"
    r"fashion (?:images?|retrieval|products?|items?|data\w*|"
    r"search|domain|clothing)|"
    r"furniture|home goods|home decor|clothing|garments?|apparel)\b",
    re.I,
)
RELATED = re.compile(
    r"\b(image retrieval|image search|shape retrieval|"
    r"fine-grained (?:visual|image) recognition)\b",
    re.I,
)


def parse_arxiv(content, tier):
    feed = ET.fromstring(content)
    entries = feed.findall("a:entry", ATOM)
    if any(
        "/api/errors" in entry.findtext("a:id", "", ATOM) for entry in entries
    ):
        raise ValueError("arXiv вернул ошибку в Atom-ответе")

    rows = []
    for entry in entries:
        url = entry.findtext("a:id", "", ATOM).strip()
        title = " ".join(entry.findtext("a:title", "", ATOM).split())
        abstract = " ".join(entry.findtext("a:summary", "", ATOM).split())
        published = entry.findtext("a:published", "", ATOM)
        authors = [
            a.text.strip()
            for a in entry.findall("a:author/a:name", ATOM)
            if a.text and a.text.strip()
        ]
        if not all((url, title, abstract, authors, published[:4].isdigit())):
            continue

        arxiv_id = re.sub(r"v[0-9]+$", "", url.split("/abs/")[-1])
        rows.append(
            {
                "doc_id": arxiv_id,
                "title": title,
                "authors": authors,
                "year": int(published[:4]),
                "abstract": abstract,
                "url": f"https://arxiv.org/abs/{arxiv_id}",
                "query_tier": tier,
            }
        )
    return rows, len(entries)


def request_page(session, params, tier):
    for attempt in range(CORPUS["attempts"]):
        time.sleep(CORPUS["interval"] * 2**attempt)
        try:
            response = session.get(
                ARXIV_ENDPOINT, params=params, timeout=CORPUS["timeout"]
            )
            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After", "")
                if retry_after.isdigit():
                    time.sleep(int(retry_after))
            response.raise_for_status()
            return parse_arxiv(response.content, tier)
        except requests.HTTPError as error:
            status = error.response.status_code
            if status < 500 and status != 429:
                raise
            failure = error
        except (requests.RequestException, ET.ParseError, ValueError) as error:
            failure = error
    raise RuntimeError(
        f"Не удалось получить страницу arXiv, запрос {tier}."
    ) from failure


def fits_scope(article):
    title = article["title"]
    text = title + " " + article["abstract"]
    if article["query_tier"] == 4:
        return bool(RELATED.search(title))
    if article["query_tier"] == 1:
        return bool(
            RETAIL.search(title) and VISION.search(text) and TASK.search(text)
        )
    title_is_fashion = re.search(r"\bfashion\b", title, re.I)
    return bool(
        (RETAIL.search(text) or title_is_fashion)
        and VISION.search(text)
        and TASK.search(text)
    )


def collect_corpus():
    categories = "(cat:cs.CV OR cat:cs.MM OR cat:cs.IR)"
    date_range = f"submittedDate:[201001010000 TO {CORPUS['stop_date']}]"
    articles = {}
    with (
        requests.Session() as session,
        tqdm(total=CORPUS["size"], unit="статья") as bar,
    ):
        session.headers.update(
            {"User-Agent": "IKEA-visual-product-search-lab/1.0"}
        )
        for tier, query in enumerate(ARXIV_QUERIES, start=1):
            start = 0
            search_query = f"{categories} AND {date_range} AND ({query})"
            while len(articles) < CORPUS["size"]:
                params = {
                    "search_query": search_query,
                    "start": start,
                    "max_results": CORPUS["page_size"],
                    "sortBy": "relevance",
                    "sortOrder": "descending",
                }
                rows, fetched = request_page(session, params, tier)
                for row in rows:
                    if row["doc_id"] in articles or not fits_scope(row):
                        continue
                    articles[row["doc_id"]] = row
                    bar.update(1)
                    if len(articles) == CORPUS["size"]:
                        return list(articles.values())

                if fetched < CORPUS["page_size"]:
                    break
                start += fetched
    raise RuntimeError(f"Собрано {len(articles)} из {CORPUS['size']} статей.")


corpus_dir = artifact_dir(
    "corpus", fingerprint(ARXIV_QUERIES, CORPUS, "scope-v1")
)
corpus = cached_rows(corpus_dir / "articles.jsonl", collect_corpus)
corpus_key = fingerprint(corpus)
corpus_frame = pd.DataFrame(corpus)
print(f"Уникальные статьи: {corpus_frame.doc_id.nunique()}")
display(corpus_frame.groupby("query_tier").size().rename("Статьи").to_frame())
display(
    corpus_frame.groupby("query_tier").head(3)[
        ["query_tier", "title", "year", "url"]
    ]
)

  0%|          | 0/300 [00:00<?, ?статья/s]

  0%|          | 1/300 [00:03<19:45,  3.97s/статья]

  1%|▏         | 4/300 [00:07<08:10,  1.66s/статья]

 21%|██        | 62/300 [00:10<00:29,  8.00статья/s]

 30%|██▉       | 89/300 [00:13<00:26,  8.08статья/s]

 49%|████▉     | 148/300 [00:17<00:13, 11.55статья/s]

... (6 outputs left out)

100%|██████████| 300/300 [00:36<00:00,  8.12статья/s]

Сохранено: data\corpus\a73a8b66ce8a66c8\articles.jsonl
Уникальные статьи: 300


,Статьи
query_tier,
1,3
2,85
3,135
4,77


,query_tier,title,year,url
0,1,3D-FUTURE: 3D Furniture shape with TextURE,2020,https://arxiv.org/abs/2009.09633
1,1,Object classification in images of Neoclassica...,2017,https://arxiv.org/abs/1703.02445
2,1,The IKEA ASM Dataset: Understanding People Ass...,2020,https://arxiv.org/abs/2007.00394
3,2,FashionLens: Toward Versatile Fashion Image Re...,2026,https://arxiv.org/abs/2605.22552
4,2,Hierarchical Similarity Learning for Language-...,2021,https://arxiv.org/abs/2102.09375
5,2,Conversational Fashion Image Retrieval via Mul...,2021,https://arxiv.org/abs/2106.04128
88,3,Visual Fashion-Product Search at SK Planet,2016,https://arxiv.org/abs/1609.07859
89,3,Bag of Tricks for Retail Product Image Classif...,2020,https://arxiv.org/abs/2001.03992
90,3,The Design and Implementation of a Real Time V...,2019,https://arxiv.org/abs/1908.07389
223,4,You'll Never Walk Alone: A Sketch and Text Due...,2024,https://arxiv.org/abs/2403.07222


### Вывод

Собрано 300 уникальных публикаций. 223 прошли фильтры по мебели, поиску товаров и электронной коммерции. Ещё 77 описывают смежные методы поиска изображений или точного визуального распознавания. Пример работы для прикладного контекста IKEA - поиск по каталогу товаров для дома в статье [From Pixels to Purchase](https://arxiv.org/abs/2601.11769).

## 2. Препроцессинг и чанкинг

`prepare_documents` соединяет заголовок с аннотацией и сохраняет границу
между ними. `word_windows` двигает окно из 130 слов с перекрытием 25 слов.
`recursive_splitter` создаёт `RecursiveCharacterTextSplitter` с теми же
размером и целевым перекрытием. Приоритет у границ предложений, затем
используются пробелы. Разрыв между заголовком и аннотацией не имеет
отдельного приоритета, чтобы заголовок оставался вместе с началом текста.
Фактическое перекрытие зависит от границ фрагментов. `split_articles`
сохраняет идентификатор чанка, его текст и родительские метаданные.
`chunk_statistics` сводит размеры и число фрагментов по стратегиям.

In [6]:
def prepare_documents():
    return [
        dict(article, text=f"{article['title']}\n\n{article['abstract']}")
        for article in corpus
    ]


def word_windows(text):
    words = text.split()
    size = CHUNKS["size"]
    step = size - CHUNKS["overlap"]
    for offset in range(0, len(words), step):
        yield " ".join(words[offset : offset + size])
        if offset + size >= len(words):
            break


def recursive_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNKS["size"],
        chunk_overlap=CHUNKS["overlap"],
        separators=[". ", "? ", "! ", " ", ""],
        length_function=lambda text: len(text.split()),
        keep_separator="end",
    )


def split_articles(strategy, split):
    rows = []
    for article in documents:
        for order, chunk in enumerate(split(article["text"])):
            title = article["title"]
            rows.append(
                {
                    "chunk_id": f"{strategy}/{article['doc_id']}/{order}",
                    "doc_id": article["doc_id"],
                    "title": title,
                    "authors": article["authors"],
                    "year": article["year"],
                    "url": article["url"],
                    "text": chunk,
                    "has_title": title in " ".join(chunk.split()),
                }
            )
    return rows


def chunk_statistics():
    rows = []
    for strategy, chunks in chunks_by_strategy.items():
        lengths = [len(chunk["text"].split()) for chunk in chunks]
        rows.append(
            {
                "strategy": strategy,
                "chunks": len(chunks),
                "mean_per_article": len(chunks) / len(documents),
                "median_words": np.median(lengths),
                "max_words": max(lengths),
                "without_title": sum(
                    not chunk["has_title"] for chunk in chunks
                ),
            }
        )
    return pd.DataFrame(rows).set_index("strategy")


documents = cached_rows(corpus_dir / "documents.jsonl", prepare_documents)
splitters = {
    "window": word_windows,
    "recursive": recursive_splitter().split_text,
}
chunks_dir = artifact_dir("chunks", fingerprint(corpus_key, CHUNKS, "word-v2"))
chunks_by_strategy = {
    name: cached_rows(
        chunks_dir / f"{name}.jsonl", lambda: split_articles(name, split)
    )
    for name, split in splitters.items()
}
display(chunk_statistics().round(3))

Сохранено: data\corpus\a73a8b66ce8a66c8\documents.jsonl
Сохранено: data\chunks\ccd40d9bfd0aa7ca\window.jsonl
Сохранено: data\chunks\ccd40d9bfd0aa7ca\recursive.jsonl


,chunks,mean_per_article,median_words,max_words,without_title
strategy,,,,,
window,617,2.057,130.0,130,316
recursive,615,2.050,110.0,130,314


### Вывод

Скользящее окно дало 617 чанков с медианой 130 слов. Рекурсивный разрез дал 615 чанков с медианой 110 слов. Максимум в обеих стратегиях - 130 слов. Приоритет границ предложений позволяет работать с более короткими законченными фрагментами. В обеих выборках заголовки входят в содержательные чанки.

## 3. Эмбеддинги и индексы FAISS

`encode_texts` строит L2-нормализованные векторы BGE. Для поисковых
запросов добавляется инструкция из карточки модели. `load_index`
векторизует чанки каждой стратегии отдельно и записывает матрицу и
`IndexFlatIP` с тем же порядком строк. Скалярное произведение двух
нормализованных векторов равно косинусному сходству. `vector_search`
выполняет поиск пакетом и сопоставляет найденные строки с метаданными.
[Карточка BGE](https://huggingface.co/BAAI/bge-small-en-v1.5).

In [7]:
def encode_texts(texts, as_queries=False, progress=False):
    if as_queries:
        prefix = "Represent this sentence for searching relevant passages: "
        texts = [prefix + text for text in texts]
    return encoder.encode(
        texts,
        batch_size=MODELS["embedding_batch"],
        normalize_embeddings=True,
        show_progress_bar=progress,
        convert_to_numpy=True,
    ).astype(np.float32, copy=False)


def load_index(strategy, chunks):
    vectors_path = index_dir / f"{strategy}.npy"
    index_path = index_dir / f"{strategy}.faiss"
    if vectors_path.is_file() and index_path.is_file():
        print("Из кэша:", index_path)
        return faiss.read_index(str(index_path))

    print("Векторизация:", strategy)
    vectors = encode_texts([chunk["text"] for chunk in chunks], progress=True)
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    with vectors_path.with_suffix(".npy.tmp").open("wb") as stream:
        np.save(stream, vectors, allow_pickle=False)
    vectors_path.with_suffix(".npy.tmp").replace(vectors_path)
    faiss.write_index(index, str(index_path.with_suffix(".faiss.tmp")))
    index_path.with_suffix(".faiss.tmp").replace(index_path)
    return index


def vector_search(queries, top_k, strategy):
    index = indices[strategy]
    if not 0 < top_k <= index.ntotal:
        raise ValueError(f"top_k должно быть в диапазоне 1..{index.ntotal}")

    scores, positions = index.search(
        encode_texts(queries, as_queries=True), top_k
    )
    chunks = chunks_by_strategy[strategy]
    return [
        [
            dict(chunks[pos], score=float(score))
            for pos, score in zip(row, values)
        ]
        for row, values in zip(positions, scores)
    ]


encoder = SentenceTransformer(
    str(local_model(MODELS["embedding"])), device="cpu"
)
index_key = fingerprint(chunks_by_strategy, MODELS["embedding"], "cosine-v1")
index_dir = artifact_dir("retrieval/indices", index_key)
indices = {
    name: load_index(name, chunks)
    for name, chunks in chunks_by_strategy.items()
}
for name, index in indices.items():
    print(f"{name}: {index.ntotal} векторов, размерность {index.d}")

Векторизация:

 window


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   5%|▌         | 1/20 [00:01<00:26,  1.38s/it]

Batches:  10%|█         | 2/20 [00:02<00:24,  1.37s/it]

Batches:  15%|█▌        | 3/20 [00:04<00:24,  1.41s/it]

... (36 outputs left out)

Batches:  90%|█████████ | 18/20 [00:18<00:01,  1.49it/s]

Batches:  95%|█████████▌| 19/20 [00:18<00:00,  1.74it/s]

Batches: 100%|██████████| 20/20 [00:18<00:00,  1.08it/s]

window: 617 векторов, размерность 384
recursive: 615 векторов, размерность 384


## 4. Очистка запроса и функция `retrieve`

`correct_token` сохраняет замеченные в корпусе слова и защищает
аббревиатуры, смешанный регистр, цифры, дефисы и формы множественного
числа на `s` с известной основой. Для обычного неизвестного слова
допустимо единственное частотное исправление на расстоянии
Дамерау - Левенштейна 1. `clean_query` применяет эти правила и нормализует
пробелы. `build_frequencies` строит словарь по корпусу и передаёт
частоты для записи в `data/`. Эталоны оценки не участвуют в выборе
исправления. `retrieve` принимает строку запроса и глубину, поддерживает
обе стратегии и переключатель очистки.

In [8]:
TOKEN = re.compile(r"[A-Za-z0-9]+(?:[-_/][A-Za-z0-9]+)*")


def correct_token(token):
    lower = token.lower()
    if (
        not token.isalpha()
        or not (token.islower() or token.istitle())
        or len(lower) < SPELLING["min_length"]
        or lower in word_counts
        or (lower.endswith("s") and lower[:-1] in word_counts)
    ):
        return token

    matches = process.extract(
        lower,
        candidate_words,
        scorer=DamerauLevenshtein.distance,
        score_cutoff=1,
        limit=2,
    )
    if len(matches) == 1:
        replacement = matches[0][0]
        return replacement.capitalize() if token.istitle() else replacement
    return token


def clean_query(query):
    return TOKEN.sub(
        lambda match: correct_token(match[0]), " ".join(query.split())
    )


def retrieve(query: str, top_k: int, strategy="recursive", *, clean=True):
    query = clean_query(query) if clean else query
    if not query.strip():
        raise ValueError("Введите непустой запрос.")
    return vector_search([query], top_k, strategy)[0]


def build_frequencies():
    counts = Counter(
        word.lower()
        for record in documents
        for word in TOKEN.findall(record["text"])
        if word.isalpha()
    )
    return [
        {"word": word, "frequency": count}
        for word, count in sorted(counts.items())
    ]


lexicon_dir = artifact_dir(
    "retrieval/lexicon", fingerprint(corpus_key, TOKEN.pattern)
)
frequencies = cached_rows(lexicon_dir / "frequencies.jsonl", build_frequencies)
word_counts = {row["word"]: row["frequency"] for row in frequencies}
candidate_words = sorted(
    word
    for word, count in word_counts.items()
    if len(word) >= SPELLING["min_length"]
    and count >= SPELLING["min_frequency"]
)
print("Слов в словаре исправлений:", len(candidate_words))
display(
    pd.DataFrame(retrieve("retrieval of similar product images", 3))[
        ["title", "url", "score"]
    ]
)

Сохранено: data\retrieval\lexicon\dec37f3f5cbfcea1\frequencies.jsonl
Слов в словаре исправлений: 1856


,title,url,score
0,SIR: Similar Image Retrieval for Product Searc...,https://arxiv.org/abs/2009.13836,0.892858
1,Unified Vision-Language Representation Modelin...,https://arxiv.org/abs/2302.05093,0.868202
2,PRISM: Product Retrieval In Shopping Carts usi...,https://arxiv.org/abs/2509.14985,0.857993


## 5. Локальная генерация

`open_llm` загружает Qwen с фиксированной ревизией. `complete` применяет
шаблон диалога и возвращает только новые токены. SQLite-кэш хранит
ответы модели на уровне отдельных вызовов, чтобы прерванные эксперименты
можно было продолжить. Ключ включает модель, настройки, вход и seed.
`format_contexts` размечает фрагменты индексами `[S1]`, `[S2]` и т. д.
`answer_question` задаёт одинаковые ограничения длины для двух режимов.
В RAG-режиме промпт требует сослаться на фрагменты прямо в ответе.
`format_grounded_answer` проверяет идентификаторы и преобразует маркер
отсутствия сведений в точную фразу по ТЗ. `answer_question` повторяет
запрос при ошибке формата. Если после повтора ссылки проверить нельзя,
конвейер возвращает точный отказ. `citation_status` показывает наличие
и допустимость ссылок. Подтверждение утверждений проверяется по текстам
статей и в RAGAS. `read_json_object` извлекает структурированные ответы
при подготовке вопросов и оценке галлюцинаций.

In [9]:
BASE_SYSTEM = (
    "Answer the question in English in at most two concise sentences."
)
RAG_SYSTEM = (
    "Answer using only the evidence in the excerpts below. Give one or two "
    "complete English sentences. Include the requested mechanism, "
    "comparison or finding explicitly. Cite the supporting labels inline "
    "in square brackets, e.g. [S1]. If the excerpts cannot answer, output "
    "exactly NO_ANSWER. Do not add a heading or explain your reasoning."
)


def open_llm(role):
    path = local_model(MODELS[role])
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = (
        AutoModelForCausalLM.from_pretrained(
            path, dtype=dtype, local_files_only=True
        )
        .to(DEVICE)
        .eval()
    )
    return tokenizer, model


def complete(runtime, role, system, user, max_tokens, count=1):
    key = fingerprint(
        MODELS[role], system, user, max_tokens, count, SEED, "beam-v1"
    )
    with sqlite3.connect(completion_path) as database:
        value = database.execute(
            "SELECT value FROM completions WHERE key = ?", (key,)
        ).fetchone()
        if value is not None:
            return json.loads(value[0])

        tokenizer, model = runtime
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
                num_beams=count,
                num_return_sequences=count,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = output[:, inputs.input_ids.shape[1] :]
        answers = [
            text.strip()
            for text in tokenizer.batch_decode(
                new_tokens, skip_special_tokens=True
            )
        ]
        database.execute(
            "INSERT OR REPLACE INTO completions VALUES (?, ?)",
            (key, json.dumps(answers, ensure_ascii=False)),
        )
    return answers


def format_contexts(hits):
    return [
        f"[S{rank}] {hit['title']}\n{hit['text']}"
        for rank, hit in enumerate(hits, start=1)
    ]


def read_json_object(text):
    try:
        value, _ = json.JSONDecoder().raw_decode(text[text.index("{") :])
    except ValueError:
        return {}
    return value if isinstance(value, dict) else {}


def format_grounded_answer(raw, contexts):
    text = re.sub(r"^answer:\s*", "", raw.strip(), flags=re.I)
    normalized = text.strip(". `").casefold()
    if normalized in {
        ABSTENTION.casefold(),
        "no_answer",
        "not specified",
        "not provided",
        "not stated",
        "insufficient information",
        "unknown",
    }:
        return ABSTENTION
    return text if citation_status(text, contexts) == "valid_ids" else None


def answer_question(runtime, question, contexts=None):
    limit = MODELS["answer_tokens"]
    if contexts is None:
        return complete(runtime, "answer", BASE_SYSTEM, question, limit)[0]

    user = f"Question: {question}\n\nExcerpts:\n" + "\n\n".join(contexts)
    retry = RAG_SYSTEM + " End each factual sentence with its supporting [S#]."
    prompts = (RAG_SYSTEM, retry)
    for prompt in prompts:
        raw = complete(runtime, "answer", prompt, user, limit)[0]
        answer = format_grounded_answer(raw, contexts)
        if answer is not None:
            return answer
    return ABSTENTION


def citation_status(answer, contexts):
    if answer == ABSTENTION:
        return "refusal"
    links = set(re.findall(r"\[S(\d+)\]", answer))
    allowed = {str(rank) for rank in range(1, len(contexts) + 1)}
    if not links:
        return "missing"
    return "valid_ids" if links <= allowed else "invalid_ids"


completion_path = PATHS["data"] / "generation/completions.sqlite3"
completion_path.parent.mkdir(parents=True, exist_ok=True)
with sqlite3.connect(completion_path) as cache:
    cache.execute(
        "CREATE TABLE IF NOT EXISTS completions "
        "(key TEXT PRIMARY KEY, value TEXT)"
    )
cache.close()

### 5.1. Пять тестовых сценариев

`run_demonstrations` выполняет поиск и создаёт ответы на одни и те же
вопросы с контекстом и без него. Вопрос передаётся генератору в исходном
виде. Очистка применяется к поиску. `display_demonstrations` показывает
обе версии ответа и ссылки на найденные статьи. Для демонстрации выбрано
три фрагмента со стратегией `recursive`. Такой объём позволяет проверить
использование нескольких источников в коротком ответе.

In [10]:
DEMONSTRATIONS = [
    (
        "Методика",
        "Which model narrows PRISM's candidate set before "
        "pixel-level matching?",
    ),
    (
        "Сравнение",
        "How does adding a modifying caption change an "
        "image-only query for fashion retrieval?",
    ),
    (
        "Опечатки",
        "Which features does SK Planet use in regions of "
        "interest for fashion prodcut retrival?",
    ),
    (
        "В корпусе нет ответа",
        "What was IKEA's conversion rate in the private trial IKEA-R17-DX?",
    ),
    (
        "Обобщение",
        "What common approaches do papers use to retrieve "
        "visually similar products from a large catalog?",
    ),
]


def run_demonstrations():
    runtime = open_llm("answer")
    rows = []
    for scenario, question in tqdm(
        DEMONSTRATIONS, desc="Сценарии", unit="вопрос"
    ):
        hits = retrieve(question, EVALUATION["default_depth"])
        contexts = format_contexts(hits)
        rag_answer = answer_question(runtime, question, contexts)
        rows.append(
            {
                "scenario": scenario,
                "question": question,
                "cleaned_question": clean_query(question),
                "hits": hits,
                "answer_without_context": answer_question(runtime, question),
                "rag_answer": rag_answer,
                "citation_status": citation_status(rag_answer, contexts),
            }
        )
    return rows


def display_demonstrations(rows):
    for row in rows:
        lines = [
            f"#### {row['scenario']}",
            "",
            f"**Вопрос.** {row['question']}",
            "",
            f"**Поисковая строка.** {row['cleaned_question']}",
            "",
            f"**LLM без контекста.** {row['answer_without_context']}",
            "",
            f"**RAG.** {row['rag_answer']}",
            "",
            "**Источники из выдачи**",
            "",
        ]
        for rank, hit in enumerate(row["hits"], start=1):
            lines.append(
                f"- S{rank}. [{hit['title']}]({hit['url']}) "
                f"({hit['score']:.3f})"
            )
        display(Markdown("\n".join(lines)))


demo_dir = artifact_dir(
    "generation/demonstrations",
    fingerprint(
        index_key,
        MODELS["answer"],
        MODELS["answer_tokens"],
        DEMONSTRATIONS,
        EVALUATION["default_depth"],
        SPELLING,
        BASE_SYSTEM,
        RAG_SYSTEM,
        SEED,
    ),
)
demo_rows = cached_rows(demo_dir / "answers.jsonl", run_demonstrations)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
display_demonstrations(demo_rows)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:04<00:04,  4.61s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.75s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.88s/it]

Сценарии:   0%|          | 0/5 [00:00<?, ?вопрос/s]

Сценарии:  20%|██        | 1/5 [00:05<00:20,  5.19s/вопрос]

... (7 outputs left out)

#### Сравнение

**Вопрос.** How does adding a modifying caption change an image-only query for fashion retrieval?

**Поисковая строка.** How does adding a modifying caption change an image-only query for fashion retrieval?

**LLM без контекста.** Adding a modifying caption to an image-only query for fashion retrieval provides more specific information, potentially enhancing the accuracy of matching images that align with the described style or attributes.

**RAG.** Adding a modifying caption to an image-only query for fashion retrieval changes it into a more detailed and nuanced query, improving its ability to express fine details and subtle variations through natural language, as demonstrated in the introduction of the Challenging Fashion Queries (CFQ) dataset [S1].

**Источники из выдачи**

- S1. [Training and challenging models for text-guided fashion image retrieval](https://arxiv.org/abs/2204.11004) (0.799)
- S2. [Fashion Image-to-Image Translation for Complementary Item Retrieval](https://arxiv.org/abs/2408.09847) (0.775)
- S3. [Diverse-Intent Multi-Turn Fashion Image Retrieval](https://arxiv.org/abs/2607.20291) (0.773)

#### Опечатки

**Вопрос.** Which features does SK Planet use in regions of interest for fashion prodcut retrival?

**Поисковая строка.** Which features does SK Planet use in regions of interest for fashion product retrieval?

**LLM без контекста.** SK Planet uses features such as image recognition and deep learning algorithms to analyze and retrieve fashion products based on user-selected images or descriptions of desired items.

**RAG.** SK Planet uses colour and appearance features extracted in a region-of-interest (ROI) of a fashion item for visual similarity. [S1]

**Источники из выдачи**

- S1. [Visual Fashion-Product Search at SK Planet](https://arxiv.org/abs/1609.07859) (0.795)
- S2. [On the Unreasonable Effectiveness of Centroids in Image Retrieval](https://arxiv.org/abs/2104.13643) (0.742)
- S3. [From Region to Patch: Attribute-Aware Foreground-Background Contrastive Learning for Fine-Grained Fashion Retrieval](https://arxiv.org/abs/2305.10260) (0.741)

#### В корпусе нет ответа

**Вопрос.** What was IKEA's conversion rate in the private trial IKEA-R17-DX?

**Поисковая строка.** What was IKEA's conversion rate in the private trial IKEA-R17-DX?

**LLM без контекста.** The IKEA-R17-DX private trial aimed to test new store layouts and did not involve a conversion rate as its primary metric; it focused on customer experience and sales performance.

**RAG.** Информация в найденных статьях отсутствует

**Источники из выдачи**

- S1. [The IKEA ASM Dataset: Understanding People Assembling Furniture through Actions, Objects and Pose](https://arxiv.org/abs/2007.00394) (0.622)
- S2. [UniECS: Unified Multimodal E-Commerce Search Framework with Gated Cross-modal Fusion](https://arxiv.org/abs/2508.13843) (0.596)
- S3. [A weakly supervised adaptive triplet loss for deep metric learning](https://arxiv.org/abs/1909.12939) (0.593)

#### Обобщение

**Вопрос.** What common approaches do papers use to retrieve visually similar products from a large catalog?

**Поисковая строка.** What common approaches do papers use to retrieve visually similar products from a large catalog?

**LLM без контекста.** Papers often employ feature extraction techniques to represent product images and then use similarity metrics or clustering algorithms to find visually similar products within a large catalog.

**RAG.** Papers use Unified Vision-Language Representation Modeling and PRISM: Product Retrieval In Shopping Carts using Hybrid Matching to retrieve visually similar products from a large catalog. [S1], [S2]

**Источники из выдачи**

- S1. [Unified Vision-Language Representation Modeling for E-Commerce Same-Style Products Retrieval](https://arxiv.org/abs/2302.05093) (0.784)
- S2. [PRISM: Product Retrieval In Shopping Carts using Hybrid Matching](https://arxiv.org/abs/2509.14985) (0.773)
- S3. [SIR: Similar Image Retrieval for Product Search in E-Commerce](https://arxiv.org/abs/2009.13836) (0.765)

### Вывод

| Сценарий | Проверка ответов |
| --- | --- |
| Методика | RAG указал SigLIP и 35 кандидатов из аннотации PRISM. Модель без контекста назвала другую модель. |
| Сравнение | RAG объяснил, что текстовое уточнение позволяет задать небольшие изменения, и связал это с CFQ. Ответ без контекста остался общим. |
| Опечатки | После исправления двух слов поиск нашёл работу SK Planet. RAG назвал цвет и признаки внешнего вида в области интереса. |
| В корпусе нет ответа | RAG вернул точную фразу отказа. Модель без контекста приписала вымышленному испытанию цель и метрики. |
| Обобщение | RAG указал унифицированные визуально-языковые представления и гибридный поиск PRISM. В ответ не вошёл приближённый поиск ближайших соседей из статьи SIR. |

## 6. Количественная оценка поиска

### 6.1. Тестовый набор

`build_title_test` берёт заголовки одних и тех же 100 статей для обеих
стратегий. Релевантны все чанки исходной статьи, включая те, у которых
в текст фрагмента заголовок не попал. Заголовок является частью исходного
документа, поэтому высокое попадание статьи на первом месте ожидаемо.
Поиск остальных чанков проверяет, находит ли индекс части той же статьи
по смыслу, даже при отсутствии точного совпадения с заголовком.

In [11]:
def build_title_test(strategy):
    groups = defaultdict(list)
    for chunk in chunks_by_strategy[strategy]:
        groups[chunk["doc_id"]].append(chunk["chunk_id"])
    return [
        {
            "query": article["title"],
            "relevant_doc_id": article["doc_id"],
            "relevant_chunk_ids": groups[article["doc_id"]],
        }
        for article in title_articles
    ]


retrieval_dir = artifact_dir(
    "evaluation/retrieval",
    fingerprint(
        index_key,
        SPELLING,
        SEED,
        EVALUATION["title_queries"],
        EVALUATION["typos"],
        EVALUATION["retrieval_depths"],
        "titles-v1",
    ),
)
title_articles = RNG["titles"].sample(corpus, EVALUATION["title_queries"])
title_tests = {
    name: cached_rows(
        retrieval_dir / f"{name}-test.jsonl", lambda: build_title_test(name)
    )
    for name in indices
}
print("Одних и тех же статей в обеих выборках:", len(title_articles))
display(pd.DataFrame(title_tests["recursive"]).head(3))

Сохранено: data\evaluation\retrieval\14fdb31d0838869a\window-test.jsonl
Сохранено: data\evaluation\retrieval\14fdb31d0838869a\recursive-test.jsonl
Одних и тех же статей в обеих выборках: 100


,query,relevant_doc_id,relevant_chunk_ids
0,"Unitail: Detecting, Reading, and Matching in R...",2204.00298,"[recursive/2204.00298/0, recursive/2204.00298/1]"
1,Self-distilled Dynamic Fusion Network for Lang...,2405.15451,"[recursive/2405.15451/0, recursive/2405.15451/1]"
2,Shop The Look: Building a Large Scale Visual S...,2006.10866,"[recursive/2006.10866/0, recursive/2006.10866/1]"


### 6.2. Метрики поиска и сравнение стратегий

`score_retrieval` считает долю найденных релевантных чанков (Recall@k),
долю релевантных в выдаче (Precision@k), обратный ранг первого попадания
(MRR@k) и наличие хотя бы одного попадания (Hit Rate@k). При отсутствии
релевантных чанков в выдаче обратный ранг равен нулю. Среднее сходство
рассчитывается для первой позиции и всех `k` позиций. Разрыв равен
разности оценок первой и именно `k`-й позиции. При `k=1` разрыв равен
нулю. Он показывает разброс внутри выдачи и не калиброван как вероятность.

`evaluate_retrieval` очищает запросы и один раз ищет до максимальной глубины.
Из префиксов той же выдачи получаются оценки на всех глубинах. Функция
сохраняет ранги с косинусными оценками, идентификаторы релевантных чанков
без заголовка и число попаданий по ним. Сводка усредняет метрики по
запросам. Отдельная таблица показывает, сколько чанков без заголовка
поиск находит на глубине 10.

In [12]:
RETRIEVAL_METRICS = [
    "Recall",
    "Precision",
    "MRR",
    "Hit Rate",
    "Mean top-1 similarity",
    "Mean top-k similarity",
    "Mean score gap",
]


def score_retrieval(hits, relevant, depth):
    hits = hits[:depth]
    scores = [hit["score"] for hit in hits]
    ranks = [
        rank
        for rank, hit in enumerate(hits, start=1)
        if hit["chunk_id"] in relevant
    ]
    return {
        "Recall": len(ranks) / len(relevant),
        "Precision": len(ranks) / depth,
        "MRR": 1 / ranks[0] if ranks else 0.0,
        "Hit Rate": float(bool(ranks)),
        "Mean top-1 similarity": scores[0],
        "Mean top-k similarity": sum(scores) / len(scores),
        "Mean score gap": scores[0] - scores[-1],
    }


def evaluate_retrieval():
    rows = []
    depths = EVALUATION["retrieval_depths"]
    for strategy, tests in tqdm(
        title_tests.items(), desc="Оценка поиска", unit="стратегия"
    ):
        queries = [clean_query(test["query"]) for test in tests]
        rankings = vector_search(queries, max(depths), strategy)
        without_title = {
            chunk["chunk_id"]
            for chunk in chunks_by_strategy[strategy]
            if not chunk["has_title"]
        }
        for test, query, hits in zip(tests, queries, rankings):
            relevant = set(test["relevant_chunk_ids"])
            target_chunks = relevant & without_title
            matched = sum(hit["chunk_id"] in target_chunks for hit in hits)
            rows.append(
                {
                    "strategy": strategy,
                    **test,
                    "cleaned_query": query,
                    "ranking": [
                        {"chunk_id": hit["chunk_id"], "score": hit["score"]}
                        for hit in hits
                    ],
                    "without_title": sorted(target_chunks),
                    "without_title_found": matched,
                    "metrics": [
                        dict(score_retrieval(hits, relevant, depth), k=depth)
                        for depth in depths
                    ],
                }
            )
    return rows


retrieval_rows = cached_rows(
    retrieval_dir / "rankings.jsonl", evaluate_retrieval
)
retrieval_long = pd.DataFrame(
    [
        dict(values, strategy=row["strategy"])
        for row in retrieval_rows
        for values in row["metrics"]
    ]
)
retrieval_summary = retrieval_long.groupby(["strategy", "k"])[
    RETRIEVAL_METRICS
].mean()
write_rows(
    retrieval_dir / "metrics.jsonl",
    retrieval_summary.reset_index().to_dict("records"),
)
for name in indices:
    print(name)
    display(retrieval_summary.loc[name].T.round(4))

no_title_rows = pd.DataFrame(
    [
        {
            "strategy": row["strategy"],
            "query": row["query"],
            "relevant": len(row["relevant_chunk_ids"]),
            "without_title": len(row["without_title"]),
            "found@10": row["without_title_found"],
        }
        for row in retrieval_rows
    ]
)
no_title_summary = no_title_rows.groupby("strategy")[
    ["relevant", "without_title", "found@10"]
].sum()
display(no_title_summary)
display(
    no_title_rows.query("without_title > `found@10`")
    .groupby("strategy")
    .head(2)
)

Оценка поиска:   0%|          | 0/2 [00:00<?, ?стратегия/s]

Оценка поиска:  50%|█████     | 1/2 [00:00<00:00,  1.66стратегия/s]

Оценка поиска: 100%|██████████| 2/2 [00:01<00:00,  1.66стратегия/s]

Оценка поиска: 100%|██████████| 2/2 [00:01<00:00,  1.65стратегия/s]

Сохранено: data\evaluation\retrieval\14fdb31d0838869a\rankings.jsonl
window


k,1,3,5,10
Recall,0.5083,0.7167,0.7533,0.7683
Precision,0.9900,0.4867,0.3080,0.1570
MRR,0.9900,0.9950,0.9950,0.9950
Hit Rate,0.9900,1.0000,1.0000,1.0000
Mean top-1 similarity,0.9232,0.9232,0.9232,0.9232
Mean top-k similarity,0.9232,0.8744,0.8553,0.8353
Mean score gap,0.0000,0.0856,0.0993,0.1127


recursive


k,1,3,5,10
Recall,0.5033,0.7150,0.7667,0.7983
Precision,0.9700,0.4767,0.3080,0.1610
MRR,0.9700,0.9850,0.9850,0.9850
Hit Rate,0.9700,1.0000,1.0000,1.0000
Mean top-1 similarity,0.9244,0.9244,0.9244,0.9244
Mean top-k similarity,0.9244,0.8745,0.8549,0.8344
Mean score gap,0.0000,0.0890,0.1015,0.1156


,relevant,without_title,found@10
strategy,,,
recursive,205,104,60
window,208,107,56


,strategy,query,relevant,without_title,found@10
0,window,"Unitail: Detecting, Reading, and Matching in R...",2,1,0
1,window,Self-distilled Dynamic Fusion Network for Lang...,2,1,0
100,recursive,"Unitail: Detecting, Reading, and Matching in R...",2,1,0
101,recursive,Self-distilled Dynamic Fusion Network for Lang...,2,1,0


### Вывод

Скользящее окно чаще находило первый релевантный чанк на первом месте (Hit Rate@1 0,990 против 0,970). У рекурсивного разреза выше Recall@10 (0,798 против 0,768). В обеих стратегиях Hit Rate@5 достиг 1,000, чему помогает совпадение запроса с заголовком. На глубине 10 для рекурсивного разреза нашлось 60 из 104 релевантных чанков без заголовка, для скользящего окна - 56 из 107. В частности, обе стратегии пропустили фрагмент без заголовка для статьи Unitail. Рост глубины повышает полноту, но снижает долю релевантных фрагментов в выдаче.

### 6.3. Парный эксперимент с опечатками

`corrupt_title` вносит одну замену, удаление или перестановку соседних
букв. Токен выбирается среди обычных слов достаточной длины. Список
искажений составлен один раз для 40 заголовков и используется в обеих
стратегиях. Примеры должны содержать реальное изменение строки.
`build_typos` сохраняет исходную строку, вид искажения и результат
очистки. `evaluate_typos` сравнивает два режима поиска на одних и тех
же запросах и множествах релевантности. Первый режим ищет по искажённой
строке, второй применяет `clean_query` к той же строке.

In [13]:
def corrupt_title(text, operation, rng):
    words = [
        match
        for match in TOKEN.finditer(text)
        if len(match[0]) > SPELLING["min_length"]
        and match[0].isalpha()
        and (match[0].islower() or match[0].istitle())
        and len(set(match[0])) > 1
    ]
    if not words:
        return None
    match = rng.choice(words)
    word = match[0]
    if operation == "delete":
        position = rng.randrange(1, len(word))
        changed = word[:position] + word[position + 1 :]
    elif operation == "replace":
        position = rng.randrange(1, len(word))
        letters = "abcdefghijklmnopqrstuvwxyz".replace(word[position], "")
        changed = word[:position] + rng.choice(letters) + word[position + 1 :]
    else:
        pairs = [i for i in range(1, len(word) - 1) if word[i] != word[i + 1]]
        if not pairs:
            return None
        position = rng.choice(pairs)
        changed = (
            word[:position]
            + word[position + 1]
            + word[position]
            + word[position + 2 :]
        )
    return text[: match.start()] + changed + text[match.end() :]


def build_typos():
    rng = RNG["typos"]
    all_tests = title_tests["recursive"]
    operations = ("replace", "delete", "transpose")
    rows = []
    for test in rng.sample(all_tests, len(all_tests)):
        operation = operations[len(rows) % len(operations)]
        original = test["query"]
        corrupted = corrupt_title(original, operation, rng)
        if corrupted is None:
            continue
        rows.append(
            {
                "doc_id": test["relevant_doc_id"],
                "original": original,
                "operation": operation,
                "corrupted": corrupted,
                "cleaned": clean_query(corrupted),
            }
        )
        if len(rows) == EVALUATION["typos"]:
            return rows
    raise RuntimeError("Недостаточно подходящих запросов для искажения.")


def evaluate_typos():
    rows = []
    for strategy, tests in tqdm(
        title_tests.items(), desc="Опечатки", unit="стратегия"
    ):
        targets = {
            test["relevant_doc_id"]: test["relevant_chunk_ids"]
            for test in tests
        }
        for mode in ("corrupted", "cleaned"):
            rankings = vector_search(
                [row[mode] for row in typo_rows], 10, strategy
            )
            for typo, hits in zip(typo_rows, rankings):
                relevant = set(targets[typo["doc_id"]])
                rows.append(
                    {
                        "doc_id": typo["doc_id"],
                        "strategy": strategy,
                        "mode": mode,
                        "Recall@5": score_retrieval(hits, relevant, 5)[
                            "Recall"
                        ],
                        "MRR@10": score_retrieval(hits, relevant, 10)["MRR"],
                    }
                )
    return rows


typo_rows = cached_rows(retrieval_dir / "typos-test.jsonl", build_typos)
typo_details = cached_rows(
    retrieval_dir / "typo-rankings.jsonl", evaluate_typos
)
typo_summary = (
    pd.DataFrame(typo_details)
    .groupby(["strategy", "mode"])[["Recall@5", "MRR@10"]]
    .mean()
)
write_rows(
    retrieval_dir / "typo-metrics.jsonl",
    typo_summary.reset_index().to_dict("records"),
)
display(typo_summary.round(4))
typo_frame = pd.DataFrame(typo_rows)
print(
    "Строк восстановлено до эталона:",
    (typo_frame.cleaned == typo_frame.original).sum(),
)
typo_examples = pd.concat(
    [
        typo_frame[typo_frame.cleaned == typo_frame.original].head(3),
        typo_frame[typo_frame.cleaned != typo_frame.original].head(3),
    ]
)
display(typo_examples[["operation", "original", "corrupted", "cleaned"]])

Сохранено: data\evaluation\retrieval\14fdb31d0838869a\typos-test.jsonl


Опечатки:   0%|          | 0/2 [00:00<?, ?стратегия/s]

Опечатки:  50%|█████     | 1/2 [00:00<00:00,  1.62стратегия/s]

Опечатки: 100%|██████████| 2/2 [00:01<00:00,  1.57стратегия/s]

Опечатки: 100%|██████████| 2/2 [00:01<00:00,  1.58стратегия/s]

Сохранено: data\evaluation\retrieval\14fdb31d0838869a\typo-rankings.jsonl


Recall@5  MRR@10
strategy  mode                       
recursive cleaned      0.7958  0.9750
          corrupted    0.7583  0.9458
window    cleaned      0.7625  0.9875
          corrupted    0.7625  0.9750

Строк восстановлено до эталона: 29


,operation,original,corrupted,cleaned
0,replace,Visually Similar Products Retrieval for Shopsy,Vdsually Similar Products Retrieval for Shopsy,Visually Similar Products Retrieval for Shopsy
1,delete,Diffusion Augmented Retrieval: A Training-Free...,Diffusion Agmented Retrieval: A Training-Free ...,Diffusion Augmented Retrieval: A Training-Free...
2,transpose,Studio2Shop: from studio photo shoots to fashi...,Studio2Shop: from studio photo shoots to fsahi...,Studio2Shop: from studio photo shoots to fashi...
3,replace,Sketch-based Image Retrieval from Millions of ...,Sketch-based Image Retrieval from Millions of ...,Sketch-based Image Retrieval from Millions of ...
4,delete,Transformer-empowered Multi-modal Item Embeddi...,Transformer-empowered Multi-modal Item Embeddi...,Transformer-empowered Multi-modal Item Embeddi...
10,delete,A Feature Consistency Driven Attention Erasing...,A Feature Consistenc Driven Attention Erasing ...,A Feature Consistenc Driven Attention Erasing ...


### Вывод

Точно восстановлены 29 из 40 запросов. Для рекурсивного разреза Recall@5 вырос с 0,758 до 0,796, MRR@10 - с 0,946 до 0,975. У скользящего окна Recall@5 остался 0,763, а MRR@10 вырос с 0,975 до 0,988. Например, `Vdsually` исправилось на `Visually`. После удаления буквы из `Enhanced` получилось допустимое слово `Enhance`, поэтому словарь оставил его без изменений.

## 7. Оценка генерации через RAGAS

### 7.1. Вопросы и эталонные ответы

`parse_question` читает JSON из ответа генератора и проверяет точное
наличие эталонной цитаты в аннотации. Вопросы с отсылками к неназванной
работе и вопросы с несколькими ожидаемыми фактами отбрасываются.
`REVIEWED_QUESTIONS` уточняет два вопроса по содержанию эталонной цитаты.
Такие уточнения привязаны к конкретной статье и исходной формулировке.
`build_questions` проходит
статьи в порядке воспроизводимой случайной выборки. По каждой аннотации
локальная Qwen формулирует один вопрос и находит предложение или
фрагмент с ответом. Использованная цитата сохраняется как `ground_truth`,
а аннотация и `doc_id` как эталонный контекст. Уникальность проверяется
по нормализованному тексту вопроса.

Четыре контрольных вопроса содержат специально подобранные идентификаторы
частных экспериментов. `add_unanswerable` проверяет отсутствие этих
идентификаторов во всём корпусе. Ни метки, ни эталонный текст не передаются
в поиск или в генерацию ответов. Выдача по этим вопросам сохраняется
для проверки того, как модель обработала тематически близкие статьи.

In [14]:
QUESTION_SYSTEM = (
    "First select a short contiguous quote from the English abstract that "
    "states one concrete fact. Copy 8 to 55 words verbatim as evidence. "
    "Then write one English question fully answered by that exact quote. "
    "Identify the task, benchmark or method by name in the question. Avoid "
    'references such as "the paper", "this work" or "the proposed model". '
    "Ask for a single fact, not an enumeration or several details. "
    'Return one JSON object only: {"question":"...", "evidence":"..."}. '
    "Escape quotes as required in JSON."
)
VAGUE_QUESTION = re.compile(
    r"\b(?:the paper|this paper|this work|this study|the authors|"
    r"the proposed (?:model|method|approach|framework)|"
    r"the reformulated task)\b"
    r"|^what are (?:the )?(?:two|three|four|five|all)\b",
    re.I,
)
REVIEWED_QUESTIONS = {
    "1908.07389": {
        "What is the scale of image updates handled by the system?": (
            "At what catalog scale did JD's visual search support "
            "sub-second retrieval?"
        ),
    },
    "2511.00925": {
        "How does the Dynamic Multi-level Weighted Alignment Network "
        "address the issue of imbalanced samples and inconsistent "
        "training data?": (
            "What training-data limitations motivate the Dynamic "
            "Multi-level Weighted Alignment Network for sketch-based "
            "image retrieval?"
        ),
    },
}
NEGATIVE_QUESTIONS = (
    (
        "IKEA-R17-DX",
        "What was IKEA's conversion rate in the private trial IKEA-R17-DX?",
    ),
    (
        "FJORD-Z91",
        "How many product photos were in IKEA's internal FJORD-Z91 audit?",
    ),
    (
        "BIRCH-V07",
        "What was the exact GPU latency of IKEA's internal BIRCH-V07 rollout?",
    ),
    (
        "OAK-X63",
        "What threshold did IKEA choose in the confidential "
        "OAK-X63 search test?",
    ),
)


def parse_question(text, abstract):
    payload = read_json_object(text)
    try:
        question = " ".join(payload["question"].split())
        evidence = " ".join(payload["evidence"].split())
    except (KeyError, AttributeError):
        return None

    if (
        not question.endswith("?")
        or len(question.split()) < 7
        or VAGUE_QUESTION.search(question)
        or not 8 <= len(evidence.split()) <= 55
        or evidence not in abstract
    ):
        return None
    return question, evidence


def add_unanswerable(rows):
    corpus_text = " ".join(doc["text"] for doc in documents).casefold()
    for identifier, question in NEGATIVE_QUESTIONS:
        if identifier.casefold() in corpus_text:
            raise ValueError(f"Идентификатор {identifier} найден в корпусе.")
        rows.append(
            {
                "question_id": identifier,
                "question": question,
                "has_answer": False,
                "doc_id": None,
                "reference_context": None,
                "ground_truth": ABSTENTION,
            }
        )
    return rows


def build_questions():
    runtime = open_llm("answer")
    rows = []
    seen = set()
    ordered = RNG["questions"].sample(corpus, len(corpus))
    with tqdm(
        total=EVALUATION["answerable"], desc="Эталонные вопросы", unit="вопрос"
    ) as bar:
        for article in ordered:
            abstract = article["abstract"]
            text = complete(runtime, "answer", QUESTION_SYSTEM, abstract, 256)[
                0
            ]
            parsed = parse_question(text, abstract)
            if parsed is None:
                continue
            question, evidence = parsed
            reviewed = REVIEWED_QUESTIONS.get(article["doc_id"], {})
            question = reviewed.get(question, question)
            if question.casefold() in seen:
                continue
            seen.add(question.casefold())
            rows.append(
                {
                    "question_id": f"qa-{len(rows) + 1:02d}",
                    "question": question,
                    "has_answer": True,
                    "doc_id": article["doc_id"],
                    "reference_context": abstract,
                    "ground_truth": evidence,
                }
            )
            bar.update(1)
            if len(rows) == EVALUATION["answerable"]:
                return add_unanswerable(rows)
    raise RuntimeError(
        "Не удалось собрать достаточно вопросов с проверяемым эталоном."
    )


qa_dir = artifact_dir(
    "evaluation/questions",
    fingerprint(
        corpus_key,
        MODELS["answer"],
        QUESTION_SYSTEM,
        REVIEWED_QUESTIONS,
        NEGATIVE_QUESTIONS,
        EVALUATION["answerable"],
        SEED,
        ABSTENTION,
    ),
)
qa_rows = cached_rows(qa_dir / "questions.jsonl", build_questions)
display(
    pd.DataFrame(qa_rows)[
        ["question_id", "question", "ground_truth", "has_answer"]
    ]
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:05<00:05,  5.24s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.16s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.32s/it]

Эталонные вопросы:   0%|          | 0/16 [00:00<?, ?вопрос/s]

Эталонные вопросы:   6%|▋         | 1/16 [00:04<01:14,  4.94s/вопрос]

... (14 outputs left out)

Эталонные вопросы: 100%|██████████| 16/16 [01:29<00:00,  7.36s/вопрос]

Эталонные вопросы: 100%|██████████| 16/16 [01:29<00:00,  5.59s/вопрос]

Сохранено: data\evaluation\questions\450dba5aa8a4b1a6\questions.jsonl


,question_id,question,ground_truth,has_answer
0,qa-01,What is the recognition accuracy achieved by t...,"Our approach was evaluated on both datasets, a...",True
1,qa-02,What is a key characteristic of sketches that ...,"In this paper, we study a further trait of ske...",True
2,qa-03,How do vision transformers perform in instance...,"However, in instance-level image retrieval, tr...",True
3,qa-04,How does Cat2Real address the domain gap betwe...,To bridge the severe domain gap between pristi...,True
4,qa-05,At what catalog scale did JD's visual search s...,We demonstrate that our system can support rea...,True
5,qa-06,How does domain-specific weak supervision base...,We demonstrate the importance of multimodal pr...,True
6,qa-07,How does the detection-guided generative frame...,"To overcome these challenges, we introduce a d...",True
7,qa-08,How much can considering collaborative signals...,"Specifically, we propose a multitask learning ...",True
8,qa-09,How does Enriched-FineR improve upon previous ...,"To address these bottlenecks, we propose our t...",True
9,qa-10,How does the Hierarchical Similarity Learning ...,HSL first learns multi-level representations o...,True


### 7.2. Контексты и ответы на глубинах 1, 3 и 5

`collect_evaluation_answers` строит выдачу до глубины 5 один раз для
каждого вопроса. Для остальных глубин используются префиксы этой
выдачи. Метод чанкинга, модель и промпт одинаковы. На каждой глубине
сохраняются вопрос, контексты из `retrieve`, ответ генератора,
эталонный ответ и `chunk_id`. Отдельно проверяется попадание исходной
статьи и точное вхождение эталонного фрагмента в переданный контекст.

In [15]:
def collect_evaluation_answers():
    runtime = open_llm("answer")
    rows = []
    depths = EVALUATION["generation_depths"]
    with tqdm(
        total=len(qa_rows) * len(depths), desc="Ответы для RAGAS", unit="ответ"
    ) as bar:
        for sample in qa_rows:
            question = sample["question"]
            hits = retrieve(question, max(depths))
            for depth in depths:
                selected = hits[:depth]
                contexts = format_contexts(selected)
                answer = answer_question(runtime, question, contexts)
                rows.append(
                    {
                        **sample,
                        "top_k": depth,
                        "contexts": contexts,
                        "answer": answer,
                        "citation_status": citation_status(answer, contexts),
                        "chunk_ids": [hit["chunk_id"] for hit in selected],
                        "reference_retrieved": any(
                            hit["doc_id"] == sample["doc_id"]
                            for hit in selected
                        ),
                        "reference_excerpt_retrieved": sample["has_answer"]
                        and any(
                            sample["ground_truth"]
                            in " ".join(hit["text"].split())
                            for hit in selected
                        ),
                    }
                )
                bar.update(1)
    return rows


generation_dir = artifact_dir(
    "evaluation/generation",
    fingerprint(
        index_key,
        qa_rows,
        MODELS["answer"],
        MODELS["answer_tokens"],
        EVALUATION["generation_depths"],
        RAG_SYSTEM,
        SPELLING,
        SEED,
    ),
)
generation_rows = cached_rows(
    generation_dir / "responses.jsonl", collect_evaluation_answers
)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(
    "Вопросов:",
    len(qa_rows),
    "| Ответов по всем глубинам:",
    len(generation_rows),
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:04<00:04,  4.20s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.31s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.44s/it]

Ответы для RAGAS:   0%|          | 0/60 [00:00<?, ?ответ/s]

Ответы для RAGAS:   2%|▏         | 1/60 [00:00<00:24,  2.38ответ/s]

... (57 outputs left out)

Ответы для RAGAS: 100%|██████████| 60/60 [02:22<00:00,  2.04ответ/s]

Ответы для RAGAS: 100%|██████████| 60/60 [02:22<00:00,  2.38s/ответ]

Сохранено: data\evaluation\generation\476a5f5df835bd3d\responses.jsonl


Вопросов: 20 | Ответов по всем глубинам: 60


### 7.3. Локальный судья и расчёт RAGAS

`LocalJudge` передаёт промпты RAGAS локальному судье и возвращает
`LLMResult` в формате RAGAS 0.2.15. `generate_text` принимает запрос
библиотеки синхронно, `agenerate_text` обслуживает асинхронный интерфейс.
`is_finished` передаёт результат парсеру RAGAS, если модель вернула текст.
`LocalEmbeddings` реализует интерфейс RAGAS на том же BGE. Методы
`embed_query` и `embed_documents` оборачивают `encode_texts` без специальной
инструкции для поиска. Асинхронные методы используют те же эмбеддинги.

`score_ragas` вычисляет Faithfulness, Answer Relevancy, Context Precision
и Context Recall средствами RAGAS. Faithfulness требует фактических
утверждений. Контекстные метрики с эталонным ответом вычисляются только
для вопросов с ответом. Answer Relevancy сравнивает обратные вопросы
с исходным. Перед оценкой из ответа и контекстов удаляются только
служебные метки `[S1]` и их аналоги. Полный ответ со ссылками сохраняется.
Для контрольных вопросов отдельный промпт распознаёт
неподтверждённые утверждения. Ошибки оценки сохраняются рядом с записью,
неопределённые значения записываются как `null`.

`evaluate_generation` сначала замеряет оценку первого ответа и печатает
ориентир. Дальше рассчитывает остальные строки по тем же правилам.
Кэш локальной генерации повторно использует ответы судьи при совпадении
промптов. Размер выборки и число рассчитанных значений приведены в сводке.
[Описание метрик RAGAS 0.2.15](https://docs.ragas.io/en/v0.2.15/concepts/metrics/available_metrics/).

In [16]:
JUDGE_SYSTEM = (
    "Follow the evaluation instructions. Return exactly one JSON object "
    "with the requested keys and types. No markdown. Use brief explanations. "
    "Ground attribution in the text actually supplied. A specific factual "
    "answer is committal. A refusal or inability to answer is noncommittal."
)
HALLUCINATION_SYSTEM = (
    "Determine whether the response contains at least one factual claim "
    "about the subject not backed by the sources. A pure refusal or notice "
    "of insufficient information contains no such claim. Return JSON only: "
    '{"unsupported":true} or {"unsupported":false}.'
)
RAGAS_METRICS = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]


class LocalJudge(BaseRagasLLM):
    def __init__(self, runtime):
        super().__init__(
            run_config=RunConfig(timeout=600, max_retries=2, max_wait=3)
        )
        self.runtime = runtime

    def generate_text(
        self, prompt, n=1, temperature=None, stop=None, callbacks=None
    ):
        texts = complete(
            self.runtime,
            "judge",
            JUDGE_SYSTEM,
            prompt.to_string(),
            MODELS["judge_tokens"],
            count=n,
        )
        return LLMResult(
            generations=[[Generation(text=text) for text in texts]]
        )

    async def agenerate_text(
        self, prompt, n=1, temperature=None, stop=None, callbacks=None
    ):
        return self.generate_text(prompt, n, temperature, stop, callbacks)

    def is_finished(self, response):
        return all(generation.text for generation in response.generations[0])


class LocalEmbeddings(BaseRagasEmbeddings):
    def embed_query(self, text):
        return self.embed_documents([text])[0]

    def embed_documents(self, texts):
        return encode_texts(texts).tolist()

    async def aembed_query(self, text):
        return self.embed_query(text)

    async def aembed_documents(self, texts):
        return self.embed_documents(texts)


def unsupported_claim(runtime, row):
    if row["answer"] == ABSTENTION:
        return False
    contexts = "\n\n".join(row["contexts"])
    prompt = (
        f"Sources:\n{contexts}\n\n"
        f"Question: {row['question']}\nAnswer: {row['answer']}"
    )
    text = complete(runtime, "judge", HALLUCINATION_SYSTEM, prompt, 96)[0]
    value = read_json_object(text).get("unsupported")
    return value if isinstance(value, bool) else None


async def score_ragas(metrics, runtime, row):
    response = re.sub(r"\s*(?:,\s*)?\[S\d+\]", "", row["answer"]).strip()
    contexts = [re.sub(r"^\[S\d+\]\s*", "", ctx) for ctx in row["contexts"]]
    sample = SingleTurnSample(
        user_input=row["question"],
        retrieved_contexts=contexts,
        response=response,
        reference=row["ground_truth"],
    )
    scores = dict.fromkeys(RAGAS_METRICS)
    errors = {}
    is_refusal = row["answer"] == ABSTENTION
    for name, metric in metrics.items():
        if name == "faithfulness" and (
            is_refusal or not row["answer"].strip()
        ):
            continue
        if name.startswith("context_") and not row["has_answer"]:
            continue
        try:
            value = await metric.single_turn_ascore(sample, timeout=600)
            if np.isfinite(value):
                scores[name] = float(value)
            else:
                errors[name] = "RAGAS вернул NaN"
        except (
            RagasException,
            RuntimeError,
            ValueError,
            TimeoutError,
        ) as error:
            errors[name] = f"{type(error).__name__}: {error}"
    return {
        "question_id": row["question_id"],
        "top_k": row["top_k"],
        "has_answer": row["has_answer"],
        "reference_retrieved": row["reference_retrieved"],
        "reference_excerpt_retrieved": row["reference_excerpt_retrieved"],
        "abstained": is_refusal,
        "unsupported_claim": None
        if row["has_answer"]
        else unsupported_claim(runtime, row),
        "errors": errors,
        **scores,
    }


async def evaluate_generation():
    runtime = open_llm("judge")
    judge = LocalJudge(runtime)
    metrics = {
        "faithfulness": Faithfulness(llm=judge),
        "answer_relevancy": ResponseRelevancy(
            llm=judge,
            embeddings=LocalEmbeddings(),
            strictness=EVALUATION["relevancy_strictness"],
        ),
        "context_precision": LLMContextPrecisionWithReference(llm=judge),
        "context_recall": LLMContextRecall(llm=judge),
    }
    ordered = sorted(
        generation_rows,
        key=lambda row: (
            not row["has_answer"],
            row["answer"] == ABSTENTION,
            row["top_k"],
        ),
    )
    scores = []
    with tqdm(total=len(ordered), desc="RAGAS", unit="ответ") as bar:
        for position, row in enumerate(ordered):
            started = time.perf_counter()
            result = await score_ragas(metrics, runtime, row)
            scores.append(result)
            if position == 0:
                elapsed = time.perf_counter() - started
                estimate = elapsed * len(ordered) / 60
                print(
                    f"Пробная оценка: {elapsed:.1f} с | "
                    f"ориентир {estimate:.0f} мин"
                )
                print(
                    "Оценки:", {name: result[name] for name in RAGAS_METRICS}
                )
            bar.update(1)
    return scores


ragas_dir = artifact_dir(
    "evaluation/ragas",
    fingerprint(
        generation_rows,
        MODELS["embedding"],
        MODELS["judge"],
        MODELS["judge_tokens"],
        EVALUATION["relevancy_strictness"],
        JUDGE_SYSTEM,
        HALLUCINATION_SYSTEM,
        SEED,
        "ragas-0.2.15-v1",
    ),
)
ragas_path = ragas_dir / "scores.jsonl"
if ragas_path.is_file():
    ragas_rows = cached_rows(ragas_path, None)
else:
    ragas_rows = await evaluate_generation()
    write_rows(ragas_path, ragas_rows)
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print(
    "LLM-судья:",
    MODELS["judge"]["id"],
    "| Ревизия:",
    MODELS["judge"]["revision"],
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:05<00:05,  5.11s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.70s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.91s/it]

RAGAS:   0%|          | 0/60 [00:00<?, ?ответ/s]

RAGAS:   2%|▏         | 1/60 [00:31<31:21, 31.88s/ответ]

... (50 outputs left out)

RAGAS:  95%|█████████▌| 57/60 [33:38<00:16,  5.42s/ответ]

RAGAS: 100%|██████████| 60/60 [33:38<00:00,  3.50s/ответ]

RAGAS: 100%|██████████| 60/60 [33:38<00:00, 33.64s/ответ]

LLM-судья: Qwen/Qwen2.5-3B-Instruct | Ревизия: aa8e72537993ba99e69dfaafa59ed015b17504d1


### 7.4. Абляция, отказы и сопоставление с оценкой поиска

`summarize_abstention` считает долю точных отказов и число ответов
с неподтверждёнными утверждениями. Неоднозначные ответы судьи выделяются.
Его флаг оценивает наличие утверждения, поэтому отказ и галлюцинация
имеют разные критерии. Средние RAGAS считаются по вопросам с эталонным
ответом и сопровождаются числом доступных оценок по каждой глубине.
Попадание исходной статьи показывается для ответных вопросов. Качество
генерации оценивается по конкретным переданным фрагментам.

In [17]:
def summarize_abstention(scores):
    rows = []
    for depth, group in scores[~scores.has_answer].groupby("top_k"):
        total = len(group)
        correct = int(group.abstained.sum())
        known = group.unsupported_claim.dropna()
        unsupported = int(known.sum())
        rows.append(
            {
                "top_k": int(depth),
                "questions": total,
                "correct_abstentions": correct,
                "abstention_accuracy": correct / total,
                "hallucinations": unsupported,
                "judged": len(known),
                "judge_unclear": total - len(known),
                "hallucination_rate": unsupported / len(known)
                if len(known)
                else None,
            }
        )
    return rows


ragas_frame = pd.DataFrame(ragas_rows)
answerable_scores = ragas_frame[ragas_frame.has_answer]
by_depth = answerable_scores.groupby("top_k")
ragas_summary = by_depth[RAGAS_METRICS].mean()
ragas_counts = by_depth[RAGAS_METRICS].count()
reference_hits = by_depth[
    ["reference_retrieved", "reference_excerpt_retrieved"]
].sum()
abstention_rows = summarize_abstention(ragas_frame)
write_rows(
    ragas_dir / "means.jsonl",
    json.loads(ragas_summary.reset_index().to_json(orient="records")),
)
write_rows(
    ragas_dir / "counts.jsonl", ragas_counts.reset_index().to_dict("records")
)
write_rows(
    ragas_dir / "reference-hits.jsonl",
    reference_hits.reset_index().to_dict("records"),
)
write_rows(ragas_dir / "abstentions.jsonl", abstention_rows)
print("Средние RAGAS по вопросам с ответом")
display(ragas_summary.round(4))
print("Число успешно рассчитанных значений")
display(ragas_counts.join(reference_hits))
print("Вопросы без ответа")
display(pd.DataFrame(abstention_rows).set_index("top_k").round(4))

answer_frame = pd.DataFrame(generation_rows)
answer_examples = answer_frame.loc[
    ~answer_frame.has_answer
    & (answer_frame.top_k == EVALUATION["default_depth"]),
    ["question_id", "question", "answer", "citation_status"],
]
display(answer_examples)
evaluation_errors = Counter(
    metric for row in ragas_rows for metric in row["errors"]
)
print("Ошибки расчёта по метрикам:", dict(evaluation_errors))
elapsed_minutes = (time.perf_counter() - run_started) / 60
print(f"Время прогона без установки зависимостей: {elapsed_minutes:.2f} мин")

Средние RAGAS по вопросам с ответом


,faithfulness,answer_relevancy,context_precision,context_recall
top_k,,,,
1,0.5256,0.8576,0.8750,0.9271
3,0.5078,0.8715,0.8646,0.9688
5,0.5821,0.8395,0.8008,0.9479


Число успешно рассчитанных значений


,faithfulness,answer_relevancy,context_precision,context_recall,reference_retrieved,reference_excerpt_retrieved
top_k,,,,,,
1,15,16,16,16,14,12
3,15,16,16,16,16,14
5,14,16,16,16,16,14


Вопросы без ответа


,questions,correct_abstentions,abstention_accuracy,hallucinations,judged,judge_unclear,hallucination_rate
top_k,,,,,,,
1,4,4,1.0,0,4,0,0.0
3,4,4,1.0,0,4,0,0.0
5,4,4,1.0,0,4,0,0.0


,question_id,question,answer,citation_status
49,IKEA-R17-DX,What was IKEA's conversion rate in the private...,Информация в найденных статьях отсутствует,refusal
52,FJORD-Z91,How many product photos were in IKEA's interna...,Информация в найденных статьях отсутствует,refusal
55,BIRCH-V07,What was the exact GPU latency of IKEA's inter...,Информация в найденных статьях отсутствует,refusal
58,OAK-X63,What threshold did IKEA choose in the confiden...,Информация в найденных статьях отсутствует,refusal


Ошибки расчёта по метрикам: {}
Время прогона без установки зависимостей: 39.87 мин


### Вывод

При `top_k=3` получены наибольшие Answer Relevancy (0,871) и Context Recall (0,969). При `top_k=5` средняя Faithfulness выше (0,582), но релевантность и точность контекста ниже. Faithfulness усреднена по 15, 15 и 14 ответам соответственно, так как отказы не содержат проверяемых утверждений.

На каждой глубине конвейер корректно отказался для всех четырёх вопросов без ответа. Галлюцинации в этом поднаборе не обнаружены (0 из 4 на каждой глубине).

При `top_k=3` поиск вернул исходную статью для всех 16 вопросов с ответом, а полную эталонную цитату - для 14. Например, в задаче `qa-01` статья попала в выдачу, но нужное значение 93,51% отсутствовало в переданных фрагментах, поэтому конвейер отказался. Судья поставил этому контексту Recall 1,000. Попадание статьи в выдачу и оценка судьи сами по себе не гарантируют, что в переданном контексте есть ответ.